# NEXUS-AI — EXP-003A/004 Kaggle training

Controlled training for NEXUS-HSR Tiny vs its parameter-matched Transformer baseline.

**How to use (Kaggle):**
1. Upload this notebook (File → Import Notebook) or paste it into a new Kaggle notebook.
2. Settings → Accelerator → **GPU T4 x1** (one GPU; do not assume pooled VRAM).
3. Settings → Internet → **On** (needed to clone the repository).
4. Run all cells top to bottom.

**Recorded configuration (fill in before running):**
- `REPO_COMMIT`: pin to a commit hash for strict reproducibility (`main` tracks latest).
- `SEEDS`, `EPOCHS`, `BATCH_SIZE`, `DATASET_SIZE`: experiment budget.
- `LIMIT_BATCHES`: leave `None` for full training; set a small number for a smoke run first.

Per project policy, hardware, software versions, precision, seed, and effective training
configuration are recorded into the outputs. Smoke tests on PC and GPU runs are kept
distinguishable via the saved summaries.

In [ ]:
# Experiment configuration — edit before Run All
REPO_URL = "https://github.com/RokhmadHD/NEXUS-AI.git"
REPO_COMMIT = "main"  # pin a full commit hash for strict reproducibility

SEEDS = [42, 43, 44]
EPOCHS = 10
BATCH_SIZE = 16
DATASET_SIZE = 20000        # records; splits are 80/10/10
MAX_EVENTS = 3              # sequential transfer events per sample (v0.3)
DEVICE = "cuda"             # falls back to CPU in-cell if CUDA is unavailable
LIMIT_BATCHES = None        # e.g. 30 for a smoke run before the full sweep

OUTPUT_DIR = "/kaggle/working/runs"
ARTIFACTS_DIR = "/kaggle/working/artifacts"


In [ ]:
# Record hardware and software environment (required by AGENTS.md policy)
import json, os, subprocess, sys

def record(command):
    try:
        return subprocess.run(command, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as exc:
        return f"unavailable: {exc}"

import torch
environment = {
    "nvidia_smi": record(["nvidia-smi"]),
    "gpu_name": record(["bash", "-c", "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"]),
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cpu_model": record(["bash", "-c", "grep -m1 'model name' /proc/cpuinfo"]),
}
if torch.cuda.is_available():
    environment["gpu_properties"] = {
        str(i): {
            "name": torch.cuda.get_device_name(i),
            "total_memory_gb": round(torch.cuda.get_device_properties(i).total_memory / 1e9, 2),
        }
        for i in range(torch.cuda.device_count())
    }
print(json.dumps(environment, indent=2))
assert DEVICE == "cuda" and torch.cuda.is_available(), (
    "GPU T4 runtime expected; enable the accelerator or set DEVICE='cpu'"
)

In [ ]:
# Clone the repository (or reuse an existing checkout) and make it importable
# No pip install of the package is needed: the code only requires torch and
# numpy, both preinstalled on Kaggle. Imports are wired via sys.path for the
# kernel and PYTHONPATH for the isolated script subprocesses.
import os, subprocess, sys

def looks_like_repo(path):
    return os.path.isdir(path) and os.path.exists(os.path.join(path, "pyproject.toml"))

REPO = None
for candidate in ("nexus-repo", "NEXUS-AI", "/kaggle/working/NEXUS-AI"):
    if looks_like_repo(candidate):
        REPO = candidate
        break
if REPO is None:
    REPO = "nexus-repo"
    subprocess.run(["rm", "-rf", REPO], check=True)
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO], check=True)
REPO_ABS = os.path.abspath(REPO)
print("using repository directory:", REPO_ABS)

# A reused checkout may be stale: sync it to the configured commit.
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "origin"], check=True)
if REPO_COMMIT == "main":
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "--quiet", "origin/main"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "checkout", "--quiet", REPO_COMMIT], check=True)
REPO_HEAD = subprocess.run(
    ["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
os.environ["REPO_HEAD"] = REPO_HEAD
print("using nexus-ai at commit:", REPO_HEAD)

if REPO_ABS not in sys.path:
    sys.path.insert(0, REPO_ABS)
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    REPO_ABS + os.pathsep + existing_pythonpath if existing_pythonpath else REPO_ABS
)

import nexus_ai
from nexus_ai.data import VocabularyTokenizer
print("import ok; tokenizer API present:", hasattr(VocabularyTokenizer, "load"))


In [ ]:
# Generate the synthetic dataset, fit the tokenizer (train split only), validate
import subprocess, sys, json

REPO = "nexus-repo"
PY = sys.executable
DATASET_DIR = f"{REPO}/datasets/synthetic/v03_{DATASET_SIZE}"
TOKENIZER_PATH = f"{REPO}/datasets/processed/v03_{DATASET_SIZE}/tokenizer.json"

subprocess.run([PY, "-I", f"{REPO}/scripts/generate_dataset.py",
                "--output", DATASET_DIR, "--count", str(DATASET_SIZE), "--seed", "42", "--max-events", str(MAX_EVENTS)], check=True)
subprocess.run([PY, "-I", f"{REPO}/scripts/fit_tokenizer.py",
                "--train", f"{DATASET_DIR}/train.jsonl", "--output", TOKENIZER_PATH], check=True)
validation = subprocess.run(
    [PY, "-I", f"{REPO}/scripts/validate_dataset.py",
     f"{DATASET_DIR}/train.jsonl", f"{DATASET_DIR}/validation.jsonl", f"{DATASET_DIR}/test.jsonl"],
    capture_output=True, text=True, check=True,
)
tokenizer = VocabularyTokenizer.load(TOKENIZER_PATH)
print("tokenizer vocab_size:", tokenizer.vocab_size)
for record in json.loads(validation.stdout):
    print(record["path"], "records:", record["records"], "valid:", record["valid"], "errors:", len(record["errors"]))

In [ ]:
# Verify the parameter-matched budget with the fitted tokenizer
import subprocess, sys, json

result = subprocess.run(
    [sys.executable, "-I", f"{REPO}/scripts/compare_parameter_counts.py", TOKENIZER_PATH],
    capture_output=True, text=True, check=True,
)
report = json.loads(result.stdout)
print(json.dumps(report, indent=2))
assert report["parameter_matched"], "parameter budget mismatch"


## Training sweep

Both models share identical data order, tokenizer, objective, optimizer, batch size,
and update budget per seed. Only the architecture differs. Set `LIMIT_BATCHES`
(e.g. `30`) for a quick end-to-end smoke run before the full sweep.

In [ ]:
# Run the training sweep: {model} x {seed}, parallelized across GPUs.
# Multiple GPUs are NOT one pooled VRAM (project policy): each job is an
# independent run pinned to one GPU via CUDA_VISIBLE_DEVICES. With one GPU
# this degrades gracefully to sequential execution.
import os, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed

os.makedirs(OUTPUT_DIR, exist_ok=True)
device_count = torch.cuda.device_count() if DEVICE == "cuda" else 1
assert device_count >= 1, "no CUDA devices available"
print(f"using {device_count} GPU(s) for parallel independent runs")

jobs = []
for seed in SEEDS:
    for model in ("nexus", "baseline"):
        jobs.append((model, seed))

def run_job(job):
    index, (model, seed) = job
    gpu = index % device_count
    name = f"{model}_s{seed}"
    command = [
        sys.executable, "-I", f"{REPO}/scripts/train.py",
        "--model", model,
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--tokenizer", TOKENIZER_PATH,
        "--train", f"{DATASET_DIR}/train.jsonl",
        "--validation", f"{DATASET_DIR}/validation.jsonl",
        "--device", "cuda",
        "--output", f"{OUTPUT_DIR}/{name}",
    ]
    if LIMIT_BATCHES:
        command += ["--limit-batches", str(LIMIT_BATCHES)]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    started = time.time()
    subprocess.run(command, check=True, env=env)
    return name, gpu, time.time() - started

run_names = []
with ThreadPoolExecutor(max_workers=device_count) as pool:
    futures = [pool.submit(run_job, job) for job in enumerate(jobs)]
    for future in as_completed(futures):
        name, gpu, elapsed = future.result()
        run_names.append(name)
        print(f"[done] {name} on gpu{gpu}: {elapsed:.1f}s")
print("completed runs:", sorted(run_names))


In [ ]:
# Ablation and regularization runs (same sweep machinery, 2-GPU aware).
# 1. lm_only: NEXUS-HSR without aux losses (state/consistency/symbolic = 0)
# 2. dropout: both models at dropout 0.1
def run_variant(name, model, seed, extra=()):
    command = [
        sys.executable, "-I", f"{REPO}/scripts/train.py",
        "--model", model,
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--tokenizer", TOKENIZER_PATH,
        "--train", f"{DATASET_DIR}/train.jsonl",
        "--validation", f"{DATASET_DIR}/validation.jsonl",
        "--device", "cuda",
        "--output", f"{OUTPUT_DIR}/{name}",
    ] + list(extra)
    if LIMIT_BATCHES:
        command += ["--limit-batches", str(LIMIT_BATCHES)]
    return name, command

variants = []
for seed in SEEDS:
    variants.append(run_variant(
        f"nexus_lm_only_s{seed}", "nexus", seed,
        extra=("--loss-weights-json",
               '{"state_weight":0,"consistency_weight":0,"symbolic_weight":0}'),
    ))
for seed in SEEDS:
    variants.append(run_variant(f"nexus_d01_s{seed}", "nexus", seed,
                                extra=("--dropout", "0.1")))
    variants.append(run_variant(f"baseline_d01_s{seed}", "baseline", seed,
                                extra=("--dropout", "0.1")))

def run_variant_job(job):
    index, (name, command) = job
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(index % device_count))
    started = time.time()
    subprocess.run(command, check=True, env=env)
    return name, time.time() - started

variant_names = []
with ThreadPoolExecutor(max_workers=device_count) as pool:
    futures = [pool.submit(run_variant_job, job) for job in enumerate(variants)]
    for future in as_completed(futures):
        name, elapsed = future.result()
        variant_names.append(name)
        print(f"[done] {name}: {elapsed:.1f}s")
print("completed variants:", sorted(variant_names))


In [ ]:
# Evaluate every best checkpoint on the untouched test split
import subprocess, sys, json

results = []
for name in sorted(run_names + variant_names):
    if name.startswith("nexus_lm_only"):
        model = "nexus"
    elif name.startswith("nexus_d01"):
        model = "nexus"
    elif name.startswith("baseline_d01"):
        model = "baseline"
    else:
        model = name.rsplit("_s", 1)[0]
    seed = name.rsplit("_s", 1)[1]
    proc = subprocess.run(
        [sys.executable, "-I", f"{REPO}/scripts/evaluate.py",
         "--model", model,
         "--checkpoint", f"{OUTPUT_DIR}/{name}/checkpoint_best.pt",
         "--tokenizer", TOKENIZER_PATH,
         "--data", f"{DATASET_DIR}/test.jsonl",
         "--device", DEVICE],
        capture_output=True, text=True, check=True,
    )
    payload = json.loads(proc.stdout)
    metrics = payload["metrics"]
    results.append({
        "run": name, "model": model, "seed": seed,
        "test_loss": round(metrics["loss"], 4),
        "test_accuracy": round(metrics["accuracy"], 4),
        "test_perplexity": round(metrics["perplexity"], 4),
    })

try:
    import pandas as pd
    results_frame = pd.DataFrame(results)
except ImportError:
    results_frame = results
print(results_frame)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
with open(f"{ARTIFACTS_DIR}/test_results.json", "w") as stream:
    json.dump(results, stream, indent=2)
if hasattr(results_frame, "to_csv"):
    results_frame.to_csv(f"{ARTIFACTS_DIR}/test_results.csv", index=False)

In [ ]:
# Plots: per-variant test metrics and validation-loss curves.
import matplotlib.pyplot as plt

VARIANT_LABELS = {
    ("nexus", "full"): "NEXUS-HSR (full)",
    ("nexus", "lm_only"): "NEXUS-HSR (LM only)",
    ("nexus", "d01"): "NEXUS-HSR (dropout 0.1)",
    ("baseline", "full"): "Baseline",
    ("baseline", "d01"): "Baseline (dropout 0.1)",
}

def variant_of(run):
    if "_lm_only" in run:
        return "nexus", "lm_only"
    if run.startswith("nexus_d01"):
        return "nexus", "d01"
    if run.startswith("baseline_d01"):
        return "baseline", "d01"
    return ("nexus" if run.startswith("nexus") else "baseline"), "full"

grouped = {}
for row in results:
    grouped.setdefault(variant_of(row["run"]), []).append(row)

labels, mean_loss, std_loss, mean_acc, std_acc = [], [], [], [], []
for key in VARIANT_LABELS:
    rows = grouped.get(key)
    if not rows:
        continue
    losses = [r["test_loss"] for r in rows]
    accs = [r["test_accuracy"] for r in rows]
    labels.append(VARIANT_LABELS[key])
    mean_loss.append(sum(losses) / len(losses))
    std_loss.append((sum((x - sum(losses)/len(losses))**2 for x in losses) / len(losses)) ** 0.5)
    mean_acc.append(sum(accs) / len(accs))
    std_acc.append((sum((x - sum(accs)/len(accs))**2 for x in accs) / len(accs)) ** 0.5)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = range(len(labels))
axes[0].bar(x, mean_loss, yerr=std_loss, capsize=4,
            color=["#1f77b4" if "NEXUS" in l else "#ff7f0e" for l in labels])
axes[0].set_xticks(list(x), labels, rotation=20, ha="right")
axes[0].set_title("Mean test loss (± std over seeds)")
axes[0].set_ylabel("cross-entropy (lower is better)")
axes[1].bar(x, mean_acc, yerr=std_acc, capsize=4,
            color=["#1f77b4" if "NEXUS" in l else "#ff7f0e" for l in labels])
axes[1].set_xticks(list(x), labels, rotation=20, ha="right")
axes[1].set_title("Mean test accuracy (± std over seeds)")
axes[1].set_ylabel("next-token accuracy")
plt.tight_layout()
plt.savefig(f"{ARTIFACTS_DIR}/variant_bars.png", dpi=150)
plt.show()

# Validation-loss curves from training history (full and d01 runs).
fig, ax = plt.subplots(figsize=(8, 5))
for name in sorted(run_names):
    history_path = f"{OUTPUT_DIR}/{name}/history.json"
    if not os.path.exists(history_path):
        continue
    with open(history_path) as stream:
        history = json.load(stream)["history"]
    epochs = [h["epoch"] for h in history]
    ax.plot(epochs, [h["validation"]["loss"] for h in history],
            marker="o", markersize=3,
            label="NEXUS" if name.startswith("nexus") else "Baseline",
            color="#1f77b4" if name.startswith("nexus") else "#ff7f0e",
            alpha=0.5)
ax.set_xlabel("epoch")
ax.set_ylabel("validation loss")
ax.set_title("Validation loss over epochs (all seeds)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{ARTIFACTS_DIR}/validation_curves.png", dpi=150)
plt.show()
print("charts saved to artifacts/variant_bars.png and validation_curves.png")


In [ ]:
# Answer exact-match benchmark: teacher-forced prediction of the answer
# number token, grouped by event chain depth (1..MAX_EVENTS). This is the
# sharper capability measure behind the token-accuracy numbers.
import subprocess, sys, json

benchmark_rows = []
for name in sorted(run_names + variant_names):
    model = "nexus" if name.startswith("nexus") else "baseline"
    proc = subprocess.run(
        [sys.executable, "-I", f"{REPO}/scripts/benchmark_answers.py",
         "--model", model,
         "--checkpoint", f"{OUTPUT_DIR}/{name}/checkpoint_best.pt",
         "--tokenizer", TOKENIZER_PATH,
         "--data", f"{DATASET_DIR}/test.jsonl",
         "--device", "cuda"],
        capture_output=True, text=True, check=True,
    )
    report = json.loads(proc.stdout)
    benchmark_rows.append({
        "run": name,
        "overall": report["overall_exact_match"],
        **{f"depth_{d}": v for d, v in report["exact_match_by_event_count"].items()},
    })

try:
    import pandas as pd
    print(pd.DataFrame(benchmark_rows))
except ImportError:
    print(json.dumps(benchmark_rows, indent=2))
with open(f"{ARTIFACTS_DIR}/benchmark_results.json", "w") as stream:
    json.dump(benchmark_rows, stream, indent=2)

# Aggregate exact match by (variant, depth) to expose the depth trend.
def variant_of(run):
    if "_lm_only" in run: return "nexus_lm_only"
    if run.startswith("nexus_d01"): return "nexus_d01"
    if run.startswith("baseline_d01"): return "baseline_d01"
    return "nexus" if run.startswith("nexus") else "baseline"

aggregate = {}
for row in benchmark_rows:
    variant = variant_of(row["run"])
    for key, value in row.items():
        if key.startswith("depth_"):
            aggregate.setdefault((variant, key), []).append(value)
print("\\nmean exact match by (variant, depth):")
for (variant, key), values in sorted(aggregate.items()):
    print(f"  {variant:14s} {key}: {sum(values)/len(values):.4f}")


In [ ]:
# Aggregate per-model means and record the full run provenance
import json

summary = {}
for model in ("nexus", "baseline"):
    rows = [r for r in results if r["model"] == model]
    if rows:
        summary[model] = {
            "seeds": [r["seed"] for r in rows],
            "mean_test_loss": round(sum(r["test_loss"] for r in rows) / len(rows), 4),
            "mean_test_accuracy": round(sum(r["test_accuracy"] for r in rows) / len(rows), 4),
            "mean_test_perplexity": round(sum(r["test_perplexity"] for r in rows) / len(rows), 4),
        }

provenance = {
    "repo_commit": os.environ.get("REPO_HEAD", "unknown"),
    "config": {
        "seeds": SEEDS, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "dataset_size": DATASET_SIZE, "limit_batches": LIMIT_BATCHES,
        "device": DEVICE,
    },
    "environment": {k: v for k, v in environment.items() if k != "nvidia_smi"},
    "tokenizer_vocab_size": tokenizer.vocab_size,
    "per_model_means": summary,
    "note": "smoke tests do not establish learning or reasoning capability",
}
with open(f"{ARTIFACTS_DIR}/provenance.json", "w") as stream:
    json.dump(provenance, stream, indent=2)
print(json.dumps(provenance["per_model_means"], indent=2))

In [ ]:
# Package artifacts into a single downloadable zip
import shutil

archive = "/kaggle/working/nexus_ai_kaggle_results"
shutil.make_archive(archive, "zip", ARTIFACTS_DIR)
print("artifacts:")
for root, _, files in os.walk("/kaggle/working"):
    for name in files:
        path = os.path.join(root, name)
        print(f"  {path} ({os.path.getsize(path)} bytes)")